In [0]:
# ------------------------------------------------------------
# Refresh: catalog tables + views (idempotent) and readable health summary
# ------------------------------------------------------------
monitoring_view_order = ["quota_groups", "source_health", "bronze_daily", "api_usage_daily", "api_usage_hourly",
                         "silver_daily", "quality_daily", "quarantine_current", "recent_errors",
                         "table_freshness", "system_overview"]
required_log_tables = [ingestion_log_table, api_usage_table, source_blocks_table,
                       f"{catalog}.{silver_schema}.silver_run_log", f"{catalog}.{silver_schema}.silver_quality_log"]
health_icons = {"ok": "🟢", "delayed": "🟡", "errors": "🔴", "blocked": "🔴", "never_loaded": "⚪"}
health_order = {"blocked": 1, "errors": 2, "never_loaded": 3, "delayed": 4, "ok": 5}


def refresh_monitoring():
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{monitoring_schema}")
    create_api_governance_tables()                      # API usage and blocks tables always exist
    missing = [t for t in required_log_tables if not spark.catalog.tableExists(t)]
    if missing:
        raise RuntimeError(f"Monitoring needs these log tables (run bronze and silver first): {missing}")

    sources = build_source_catalog()
    contracts = load_all_entity_contracts()
    views = monitoring_view_sql(contracts)
    for name in monitoring_view_order:
        comment, select_sql = views[name]
        spark.sql(f"CREATE OR REPLACE VIEW {monitoring_object(name)} COMMENT {sql_string_literal(comment)} AS {select_sql}")
    print(f"✅ monitoring refreshed in {catalog}.{monitoring_schema}: source_catalog ({sources} sources) "
          f"+ {len(monitoring_view_order)} views")
    return monitoring_view_order


def health_reason(row):
    """Short human explanation of a source_health row."""
    health = row["health"]
    if health == "blocked":
        return f"quota group '{row['quota_group']}' blocked until {row['blocked_until']} UTC"
    if health == "never_loaded":
        return "no successful chunk yet"
    if health == "errors":
        return f"{row['pending_error_chunks']} chunk(s) in error waiting for retry"
    if health == "delayed":
        parts = []
        if row["days_behind"] is not None and row["days_behind"] > 0:
            parts.append(f"{row['days_behind']} day(s) behind expected {row['expected_data_until']}")
        if row["pending_deferred_chunks"]:
            parts.append(f"{row['pending_deferred_chunks']} chunk(s) deferred by API budget")
        return ", ".join(parts) or "behind schedule"
    return f"data loaded until {row['data_loaded_until']}"


def print_source_health():
    """Readable one-line-per-source summary of monitoring_ai.source_health (worst first)."""
    rows = spark.table(monitoring_object("source_health")).collect()
    rows = sorted(rows, key=lambda r: (health_order.get(r["health"], 9), r["source_name"]))
    width = max([len(r["source_name"]) for r in rows] + [6])
    print("Source health")
    print("-" * (width + 60))
    for r in rows:
        icon = health_icons.get(r["health"], "❔")
        print(f"{icon} {r['source_name']:<{width}}  {r['health']:<12}  {health_reason(r)}")
    blocked = [r["source_name"] for r in rows if r["health"] == "blocked"]
    issues = [r["source_name"] for r in rows if r["health"] != "ok"]
    print("-" * (width + 60))
    print(f"Blocked: {', '.join(blocked) if blocked else 'none'}  ·  "
          f"With issues: {len(issues)} of {len(rows)}")
    return rows


def sql_string_literal(text):
    return "'" + str(text).replace("\\", "\\\\").replace("'", "\\'") + "'"

In [0]:
# ------------------------------------------------------------
# View definitions (SQL). Logs: bronze_ingestion_log, bronze_api_usage, bronze_source_blocks,
# silver_run_log, silver_quality_log, silver_quarantine_summary
# ------------------------------------------------------------
def monitoring_view_sql(contracts):
    """{view name: (comment, select sql)}"""
    bronze_log = ingestion_log_table
    silver_log = silver_object("silver_run_log")
    quality_log = silver_object("silver_quality_log")
    quarantine_summary = silver_object("silver_quarantine_summary")
    m = monitoring_object

    views = {}

    views["quota_groups"] = ("API limits and budget of every quota group", f"""
        SELECT quota_group, max(per_minute) AS per_minute, max(per_hour) AS per_hour, max(per_day) AS per_day,
               max(safety_margin) AS safety_margin,
               max(per_hour) * max(safety_margin) AS hourly_budget,
               max(per_day) * max(safety_margin) AS daily_budget,
               collect_set(source_name) AS sources
        FROM {m('source_catalog')}
        GROUP BY quota_group""")

    views["source_health"] = ("Health of every bronze source: last run, data loaded until, delay, pending chunks, blocks", f"""
        WITH chunk_attempts AS (
            SELECT source_name, endpoint_name, chunk_start, chunk_end, status,
                   row_number() OVER (PARTITION BY source_name, endpoint_name, chunk_start
                                      ORDER BY ingested_at DESC) AS attempt_rank
            FROM {bronze_log}
            WHERE chunk_start IS NOT NULL AND status IN ('ok', 'error', 'deferred')
        ),
        chunk_state AS (
            SELECT source_name,
                   max(CASE WHEN status = 'ok' THEN chunk_end END) AS last_ok_chunk_end,
                   sum(CASE WHEN status = 'error' THEN 1 ELSE 0 END) AS pending_error_chunks,
                   sum(CASE WHEN status = 'deferred' THEN 1 ELSE 0 END) AS pending_deferred_chunks
            FROM chunk_attempts WHERE attempt_rank = 1
            GROUP BY source_name
        ),
        last_run AS (
            SELECT source_name, max(ingested_at) AS last_run_at FROM {bronze_log} GROUP BY source_name
        ),
        active_blocks AS (
            SELECT quota_group, max(blocked_until) AS blocked_until FROM {source_blocks_table}
            WHERE unblocked_at IS NULL AND blocked_until > current_timestamp()
            GROUP BY quota_group
        ),
        expected AS (
            SELECT *,
                   CASE unit WHEN 'day' THEN date_add(current_date(), end_offset)
                             WHEN 'month' THEN last_day(add_months(current_date(), end_offset))
                             WHEN 'year' THEN make_date(year(current_date()) + end_offset, 12, 31)
                             ELSE current_date() END AS expected_data_until
            FROM {m('source_catalog')}
        )
        SELECT e.source_name, e.provider, e.quota_group, e.unit,
               r.last_run_at, s.last_ok_chunk_end AS data_loaded_until, e.expected_data_until,
               datediff(e.expected_data_until, s.last_ok_chunk_end) AS days_behind,
               coalesce(s.pending_error_chunks, 0) AS pending_error_chunks,
               coalesce(s.pending_deferred_chunks, 0) AS pending_deferred_chunks,
               b.blocked_until,
               CASE WHEN b.blocked_until IS NOT NULL THEN 'blocked'
                    WHEN s.last_ok_chunk_end IS NULL THEN 'never_loaded'
                    WHEN coalesce(s.pending_error_chunks, 0) > 0 THEN 'errors'
                    WHEN datediff(e.expected_data_until, s.last_ok_chunk_end) > e.tolerance_days
                         OR coalesce(s.pending_deferred_chunks, 0) > 0 THEN 'delayed'
                    ELSE 'ok' END AS health
        FROM expected e
        LEFT JOIN chunk_state s ON s.source_name = e.source_name
        LEFT JOIN last_run r ON r.source_name = e.source_name
        LEFT JOIN active_blocks b ON b.quota_group = e.quota_group""")

    views["bronze_daily"] = ("Bronze chunks, bytes and runs per day, source, endpoint and status", f"""
        SELECT to_date(ingested_at) AS day, source_name, endpoint_name, status,
               count(*) AS chunks, coalesce(sum(bytes_written), 0) AS bytes_written, count(DISTINCT run_id) AS runs
        FROM {bronze_log}
        WHERE chunk_start IS NOT NULL
        GROUP BY ALL""")

    views["api_usage_daily"] = ("API usage per quota group and day vs the daily limit and budget", f"""
        SELECT to_date(u.called_at) AS day, u.quota_group, u.caller,
               count(*) AS http_calls, round(sum(u.cost), 1) AS provider_calls,
               sum(CASE WHEN u.http_status = 429 THEN 1 ELSE 0 END) AS rate_limited_calls,
               max(q.per_day) AS daily_limit, max(q.daily_budget) AS daily_budget
        FROM {api_usage_table} u
        LEFT JOIN {m('quota_groups')} q ON q.quota_group = u.quota_group
        GROUP BY ALL""")

    views["api_usage_hourly"] = ("API usage per quota group and hour vs the hourly limit and budget", f"""
        SELECT date_trunc('hour', u.called_at) AS hour_utc, u.quota_group,
               count(*) AS http_calls, round(sum(u.cost), 1) AS provider_calls,
               sum(CASE WHEN u.http_status = 429 THEN 1 ELSE 0 END) AS rate_limited_calls,
               max(q.per_hour) AS hourly_limit, max(q.hourly_budget) AS hourly_budget
        FROM {api_usage_table} u
        LEFT JOIN {m('quota_groups')} q ON q.quota_group = u.quota_group
        GROUP BY ALL""")

    views["silver_daily"] = ("Silver rows read, valid, quarantined, inserted, updated and deleted per mapping, day and scope", f"""
        SELECT to_date(finished_at) AS day, mapping, entity, source_name, scope, status,
               count(*) AS batches, coalesce(sum(bronze_files), 0) AS bronze_files,
               coalesce(sum(`rows`), 0) AS rows_read, coalesce(sum(valid_rows), 0) AS valid_rows,
               coalesce(sum(quarantined_rows), 0) AS quarantined_rows,
               coalesce(sum(rows_inserted), 0) AS rows_inserted, coalesce(sum(rows_updated), 0) AS rows_updated,
               coalesce(sum(rows_deleted), 0) AS rows_deleted
        FROM {silver_log}
        GROUP BY ALL""")

    views["quality_daily"] = ("Silver quality rule results per day, mapping, rule and column", f"""
        SELECT to_date(logged_at) AS day, mapping, entity, rule, column_name, action,
               sum(evaluated) AS evaluated, sum(failed) AS failed
        FROM {quality_log}
        GROUP BY ALL""")

    if spark.catalog.tableExists(quarantine_summary):
        quarantine_select = f"SELECT * FROM {quarantine_summary}"
    else:
        quarantine_select = ("SELECT CAST(NULL AS STRING) AS entity, CAST(NULL AS STRING) AS source_name, "
                             "CAST(NULL AS STRING) AS rule, CAST(0 AS BIGINT) AS failed_rows, "
                             "CAST(NULL AS TIMESTAMP) AS first_quarantined_at, "
                             "CAST(NULL AS TIMESTAMP) AS last_quarantined_at WHERE false")
    views["quarantine_current"] = ("Rows currently in quarantine per entity, data source and failed rule", quarantine_select)

    views["recent_errors"] = ("Errors of the whole platform: bronze chunks, silver batches and API blocks", f"""
        SELECT 'bronze' AS layer, ingested_at AS occurred_at, source_name, endpoint_name AS object_name,
               status, error_message, run_id
        FROM {bronze_log} WHERE status IN ('error', 'failed')
        UNION ALL
        SELECT 'silver' AS layer, finished_at AS occurred_at, source_name, mapping AS object_name,
               status, error_message, run_id
        FROM {silver_log} WHERE status IN ('fail', 'error', 'invalid')
        UNION ALL
        SELECT 'api' AS layer, blocked_at AS occurred_at, quota_group AS source_name, CAST(NULL AS STRING) AS object_name,
               'blocked' AS status, reason AS error_message, CAST(NULL AS STRING) AS run_id
        FROM {source_blocks_table}""")

    freshness_parts = []
    for contract in sorted(contracts.values(), key=lambda c: c["entity"]):
        entity = contract["entity"]
        table = silver_object(entity)
        if not spark.catalog.tableExists(table):
            continue
        time_column = time_column_of(contract)
        latest = f"CAST(max(`{time_column}`) AS TIMESTAMP)" if time_column else "CAST(NULL AS TIMESTAMP)"
        is_entity = contract["table_type"] == "entity"
        quarantine = silver_object(f"{entity}_quarantine")
        quarantined = (f"(SELECT count(*) FROM {quarantine})"
                       if is_entity and spark.catalog.tableExists(quarantine) else "CAST(0 AS BIGINT)")
        sources = "count(DISTINCT source_name)" if is_entity else "CAST(1 AS BIGINT)"
        freshness_parts.append(
            f"SELECT '{entity}' AS table_name, '{contract['table_type']}' AS table_type, "
            f"count(*) AS rows_count, {sources} AS data_sources, {latest} AS latest_data_utc, "
            f"max(processed_at) AS last_changed_at, {quarantined} AS quarantined_rows FROM {table}")
    views["table_freshness"] = ("Every silver table: rows, data sources, latest data, last change and quarantined rows",
                                " UNION ALL ".join(freshness_parts) if freshness_parts else
                                "SELECT CAST(NULL AS STRING) AS table_name WHERE false")

    views["system_overview"] = ("One row with the headline indicators of the platform", f"""
        SELECT
          (SELECT max(ingested_at) FROM {bronze_log}) AS last_bronze_activity,
          (SELECT max(finished_at) FROM {silver_log}) AS last_silver_activity,
          (SELECT count(*) FROM {m('source_health')}) AS sources,
          (SELECT count(*) FROM {m('source_health')} WHERE health = 'ok') AS sources_ok,
          (SELECT count(*) FROM {m('source_health')} WHERE health <> 'ok') AS sources_with_issues,
          (SELECT count(*) FROM {m('source_health')} WHERE health = 'blocked') AS sources_blocked,
          (SELECT coalesce(sum(quarantined_rows), 0) FROM {m('table_freshness')}) AS quarantined_rows,
          (SELECT count(*) FROM {m('recent_errors')}
             WHERE occurred_at >= current_timestamp() - INTERVAL 24 HOURS) AS errors_last_24h,
          (SELECT coalesce(sum(valid_rows), 0) FROM {m('silver_daily')}
             WHERE day = current_date()) AS silver_rows_today""")
    return views

In [0]:
# ------------------------------------------------------------
# Refresh: catalog tables + views (idempotent)
# ------------------------------------------------------------
monitoring_view_order = ["quota_groups", "source_health", "bronze_daily", "api_usage_daily", "api_usage_hourly",
                         "silver_daily", "quality_daily", "quarantine_current", "recent_errors",
                         "table_freshness", "system_overview"]
required_log_tables = [ingestion_log_table, api_usage_table, source_blocks_table,
                       f"{catalog}.{silver_schema}.silver_run_log", f"{catalog}.{silver_schema}.silver_quality_log"]


def refresh_monitoring():
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{monitoring_schema}")
    create_api_governance_tables()                      # API usage and blocks tables always exist
    missing = [t for t in required_log_tables if not spark.catalog.tableExists(t)]
    if missing:
        raise RuntimeError(f"Monitoring needs these log tables (run bronze and silver first): {missing}")

    sources = build_source_catalog()
    contracts = load_all_entity_contracts()
    views = monitoring_view_sql(contracts)
    for name in monitoring_view_order:
        comment, select_sql = views[name]
        spark.sql(f"CREATE OR REPLACE VIEW {monitoring_object(name)} COMMENT {sql_string_literal(comment)} AS {select_sql}")
    print(f"✅ monitoring refreshed in {catalog}.{monitoring_schema}: source_catalog ({sources} sources) "
          f"+ {len(monitoring_view_order)} views")
    return monitoring_view_order


def sql_string_literal(text):
    return "'" + str(text).replace("\\", "\\\\").replace("'", "\\'") + "'"